# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# For easy access, we'll serialize to JSON
metadata_json = dataset.metadata.to_json()

# Display dataset name and description
print("Dataset Name:\n", metadata_json.get('name'))
print("\nDescription:\n", metadata_json.get('description'))

## 2. Data Overview

Review available record sets, fields, and their IDs.

Entities are referenced by their `@id` according to the Croissant schema.

In [ ]:
# Inspect the available record sets
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        print(f"  Description: {rs.get('description', 'N/A')}")
        print(f"  Fields:")
        for field in rs.get('fields', []):
            print(f"    Field @id: {field['@id']} (Name: {field.get('name', 'N/A')})")
        print("")

# For demonstration, try to print first record from each RecordSet:
for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nSample records for RecordSet @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            print(json.dumps(records[0], indent=2))
        else:
            print("No records found.")
    except Exception as e:
        print(f"Error retrieving records: {e}")

## 3. Data Extraction

Load data from specific record sets into DataFrames for analysis.

All references use their unique `@id` fields.

We extract all tabular record sets into pandas DataFrames.

In [ ]:
# Collect all record set @id's
record_sets_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

# Load each record set as DataFrame, keyed by @id
for record_set_id in record_sets_ids:
    records_list = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records_list)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet @id: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"Head:")
    print(df.head())

# For further analysis, select the main clinical record set (by @id)
if record_sets_ids:
    main_record_set_id = record_sets_ids[0]
    print(f"\nMain record set selected for EDA: {main_record_set_id}")
    main_df = dataframes[main_record_set_id]
else:
    main_record_set_id = None
    main_df = pd.DataFrame()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All references use their unique `@id` fields extracted above.

For this demo, select a numeric field and a grouping field from available columns.

In [ ]:
# Define IDs based on available columns
# You can find exact @id by inspecting the RecordSet fields in Section 2 above
numeric_fields = [col for col in main_df.columns if main_df[col].dtype in ['int64', 'float64']]
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # Pick first numeric field
else:  # fallback
    numeric_field_id = main_df.columns[0]
print(f"Numeric field selected: {numeric_field_id}")

# Filtering: demonstrate with a threshold
threshold = 10
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization
normalized_col = f"{numeric_field_id}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, normalized_col]].head())

# Group by categorical field (try to group by first non-numeric field)
categorical_fields = [col for col in main_df.columns if main_df[col] not in numeric_fields]
group_field_id = None
for col in main_df.columns:
    if main_df[col].dtype == 'object' and col != numeric_field_id:
        group_field_id = col
        break

if group_field_id:
    print(f"Grouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No categorical fields found for grouping.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

Below is a demonstration using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If grouping field is available, boxplot by group
if group_field_id:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()


## 6. Conclusion

This notebook demonstrates loading, exploring, and analyzing a biomedical dataset defined by a Croissant schema using the `mlcroissant` library. Key steps included:
- Loading metadata and inspecting available record sets and fields (referenced by `@id`).
- Extracting tabular data for analysis.
- Filtering and normalizing records, grouping by categorical attributes.
- Visualizing distributions and relationships between fields.

The FAIR^2 dataset enables clinical, anatomical, and molecular investigation in colorectal cancer survivors, supporting fair and reproducible biomedical research. For further analysis, refer to the Croissant schema documentation and use field `@id` consistently for data operations.